# Exercise 10 example: a 2D CNN for miRNA–target pairs

Use the Hejret 2023 data to predict whether a miRNA and target-site sequence form an observed binding pair. This example accompanies the [group project brief](exercise_10.md); you can adapt its representation and model for your project.

Represent all pairs of positions in the two sequences as a 2D grid. Use 16 channels for the ordered nucleotide pairs AA, AC, …, TT, plus one padding channel. Train two convolutional layers and a fully connected hidden layer followed by a binary output. All model weights are learned.

Reserve 20% per label from the official training file for validation. Select the epoch using validation loss, then evaluate once on the official test file. **Run all includes test evaluation.** PyTorch is normally available in Colab; for local use, install `requirements.txt`.


In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
max_epochs = 15
batch_size = 64
learning_rate = 0.001
threshold = 0.5
mirna_length = 28
target_length = 50
output_dir = Path('exercise_10_output')

torch.manual_seed(seed)
torch.set_num_threads(2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Load the training data from Zenodo

Use miRBench's processed `AGO2_CLASH_Hejret2023`, version 7, from [Zenodo record 20540907](https://zenodo.org/records/20540907). `noncodingRNA` contains the miRNA sequence; `gene` contains the 50-nt target-site sequence. RNA sequences are stored with T rather than U.

`label=1` is an observed AGO2-CLASH binding pair; `label=0` is a synthetic negative pair. These are not measurements of repression strength. Load only the two sequences and label. Other columns, especially `Nunique`, must not enter the predictor because they can reveal the answer.

Keep the official test file separate and download it only in the final evaluation cell. This is the benchmark's existing chromosome-based holdout, not a holdout of unseen miRNA families.


In [ ]:
data_url = 'https://zenodo.org/records/20540907/files/'
input_columns = ['noncodingRNA', 'gene', 'label']
benchmark_train = pd.read_csv(
    data_url + 'AGO2_CLASH_Hejret2023_train.tsv.gz?download=1',
    sep='	', compression='gzip', usecols=input_columns,
)
assert benchmark_train['noncodingRNA'].str.fullmatch(f'[ACGT]{{1,{mirna_length}}}').all()
assert benchmark_train['gene'].str.fullmatch(f'[ACGT]{{{target_length}}}').all()
assert set(benchmark_train['label']) == {0, 1}
assert not benchmark_train.duplicated(['noncodingRNA', 'gene']).any()
print('Official training examples:', len(benchmark_train))
print('miRNA lengths:', benchmark_train['noncodingRNA'].str.len().min(),
      'to', benchmark_train['noncodingRNA'].str.len().max())
print(benchmark_train['label'].value_counts())
benchmark_train.head()


## 2. Reserve 20% per label for validation

Use the remaining 80% for training. Keep this split fixed across epochs. Only training batches update weights; validation selects the epoch. This split is stratified by label, not by miRNA family or target similarity.


In [ ]:
validation_data = benchmark_train.groupby('label').sample(frac=0.20, random_state=seed)
train_data = benchmark_train.drop(validation_data.index)
y_train = train_data['label'].to_numpy()
y_validation = validation_data['label'].to_numpy()
print('Training:', len(train_data))
print('Validation:', len(validation_data))


## 3. Encode the two sequences as a grid

Rows correspond to miRNA positions; columns correspond to target positions. At cell (i,j), identify the ordered pair `(miRNA[i], target[j])`. With A=0, C=1, G=2, T=3, its channel ID is `4 * miRNA_base + target_base`. Thus A/G activates channel AG, while G/A activates channel GA.

Keep both sequences in their supplied 5′→3′ orientation. This is an all-pairs representation, not a supplied alignment, binding map, or rule about complementary bases. The network can learn local patterns of pair types.

Right-pad miRNAs to a fixed length of 28. Every cell in a padded row activates channel 16 (`PAD`), instead of a nucleotide-pair channel. Do not truncate longer sequences silently. Targets already have length 50. Each cell activates exactly one of 17 channels.

Store one small integer per cell and create one-hot channels for each mini-batch. The CNN input has shape **batch × 17 × 28 × 50**. This avoids storing the full dataset as large floating-point one-hot tensors.


In [ ]:
channel_names = [a + b for a in 'ACGT' for b in 'ACGT'] + ['PAD']
print(pd.DataFrame(np.array(channel_names[:16]).reshape(4, 4),
                   index=list('ACGT'), columns=list('ACGT')))

def encode_pairs(data):
    base_ids = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    codes = np.full((len(data), mirna_length, target_length), 16, dtype=np.uint8)
    for row, (mirna, target) in enumerate(data[['noncodingRNA', 'gene']].itertuples(index=False, name=None)):
        if not 1 <= len(mirna) <= mirna_length or len(target) != target_length:
            raise ValueError('Unexpected sequence length; do not silently truncate inputs.')
        mirna_ids = np.array([base_ids[base] for base in mirna], dtype=np.uint8)
        target_ids = np.array([base_ids[base] for base in target], dtype=np.uint8)
        codes[row, :len(mirna), :] = 4 * mirna_ids[:, None] + target_ids[None, :]
    return torch.from_numpy(codes)

def one_hot_pairs(codes):
    return F.one_hot(codes.long(), num_classes=17).permute(0, 3, 1, 2).float()

train_codes = encode_pairs(train_data)
validation_codes = encode_pairs(validation_data)
train_loader = DataLoader(
    TensorDataset(train_codes, torch.tensor(y_train, dtype=torch.float32)),
    batch_size=batch_size, shuffle=True,
)
validation_loader = DataLoader(
    TensorDataset(validation_codes, torch.tensor(y_validation, dtype=torch.float32)),
    batch_size=batch_size, shuffle=False,
)
print('Compact training grids:', tuple(train_codes.shape))
print('One-hot batch:', tuple(one_hot_pairs(train_codes[:batch_size]).shape))
print('Example miRNA:', train_data.iloc[0]['noncodingRNA'])
print('Example target:', train_data.iloc[0]['gene'])
plt.figure(figsize=(11, 5))
plt.imshow(train_codes[0].numpy(), aspect='auto', cmap=plt.get_cmap('tab20', 17), vmin=-0.5, vmax=16.5)
plt.colorbar(ticks=range(17)).ax.set_yticklabels(channel_names)
plt.xlabel('Target position (0-based)')
plt.ylabel('miRNA position (0-based)')
plt.title('Pair-channel IDs before one-hot encoding')
plt.tight_layout()
plt.show()


## 4. Build a small 2D CNN

| Layer | Output per example |
| --- | --- |
| Conv2d: 17 → 16 channels, 3 × 3 kernel, padding 1; ReLU | 16 × 28 × 50 |
| Max pooling: 2 × 2 | 16 × 14 × 25 |
| Conv2d: 16 → 32 channels, 3 × 3 kernel, padding 1; ReLU | 32 × 14 × 25 |
| Max pooling: 2 × 2; flatten | 2,688 |
| Fully connected: 2,688 → 32; ReLU; dropout 0.3 | 32 |
| Linear classifier: 32 → 1 | 1 logit |

Convolution moves across both sequence axes. All layers start with random weights. `BCEWithLogitsLoss` combines sigmoid and binary cross-entropy stably; apply sigmoid separately when reporting probabilities.


In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(17, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * (mirna_length // 4) * (target_length // 4), 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 1),
).to(device)
loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate, weight_decay=0.0001)
print(cnn)
print('Trainable parameters:', sum(p.numel() for p in cnn.parameters() if p.requires_grad))


## 5. Train and select the epoch using validation loss

For each mini-batch, clear gradients, calculate logits and loss, backpropagate, and update weights with Adam. Convert compact pair IDs to one-hot channels on the chosen device. Disable dropout and gradient tracking for validation. Restore the checkpoint with the lowest validation loss.


In [ ]:
training_losses = []
validation_losses = []
best_validation_loss = np.inf
for epoch in range(1, max_epochs + 1):
    cnn.train()
    total_loss = 0.0
    for codes, labels in train_loader:
        inputs = one_hot_pairs(codes.to(device))
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = cnn(inputs).squeeze(1)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    training_loss = total_loss / len(train_data)

    cnn.eval()
    total_loss = 0.0
    with torch.no_grad():
        for codes, labels in validation_loader:
            logits = cnn(one_hot_pairs(codes.to(device))).squeeze(1)
            loss = loss_function(logits, labels.to(device))
            total_loss += loss.item() * len(labels)
    validation_loss = total_loss / len(validation_data)
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_state = deepcopy(cnn.state_dict())
    print(f'Epoch {epoch}: training={training_loss:.4f}, validation={validation_loss:.4f}', flush=True)

cnn.load_state_dict(best_state)
cnn.eval()
print('Selected epoch:', best_epoch)
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_epochs + 1), training_losses, label='Training (with dropout)')
plt.plot(range(1, max_epochs + 1), validation_losses, label='Validation')
plt.axvline(best_epoch, color='black', linestyle='--', label='Selected epoch')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Evaluate validation predictions

Use the familiar eight metrics and threshold 0.5. The prediction function accepts a model and batches of pair IDs; the scoring function accepts only labels and probabilities. Save validation predictions and reference labels separately, using IDs based on original training-file rows. These files can help you develop your own project evaluator without opening the test results.


In [ ]:
def predict_probabilities(model, loader):
    model.eval()
    batches = []
    with torch.no_grad():
        for codes, _ in loader:
            logits = model(one_hot_pairs(codes.to(device))).squeeze(1)
            batches.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(batches)

def calculate_metrics(labels, probabilities):
    predictions = (probabilities >= threshold).astype(int)
    return pd.Series({
        'accuracy': accuracy_score(labels, predictions),
        'balanced_accuracy': balanced_accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
        'roc_auc': roc_auc_score(labels, probabilities),
        'average_precision': average_precision_score(labels, probabilities),
        'log_loss': log_loss(labels, probabilities),
    })

def plot_evaluation(labels, probabilities, name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    RocCurveDisplay.from_predictions(labels, probabilities, ax=axes[0])
    axes[0].plot([0, 1], [0, 1], 'k--')
    PrecisionRecallDisplay.from_predictions(labels, probabilities, ax=axes[1])
    axes[1].axhline(np.mean(labels), color='black', linestyle='--')
    ConfusionMatrixDisplay.from_predictions(labels, probabilities >= threshold,
                                           labels=[0, 1], ax=axes[2], cmap='Blues', colorbar=False)
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

validation_probabilities = predict_probabilities(cnn, validation_loader)
validation_results = calculate_metrics(y_validation, validation_probabilities)
print(validation_results.round(4).to_string())
plot_evaluation(y_validation, validation_probabilities, 'Validation: 2D CNN')
output_dir.mkdir(parents=True, exist_ok=True)
validation_ids = 'train_' + validation_data.index.astype(str)
pd.DataFrame({'sample_id': validation_ids, 'probability': validation_probabilities}).to_csv(
    output_dir / 'validation_predictions.csv', index=False)
pd.DataFrame({'sample_id': validation_ids, 'label': y_validation}).to_csv(
    output_dir / 'validation_answers.csv', index=False)


## 7. STOP before final test evaluation

Finish selecting the model and epoch using validation data first. Then download the unchanged official test file and evaluate the selected weights once. Do not change settings based on these results. The test split contains overlapping miRNAs and related target sequences, so performance does not demonstrate generalization to all unseen miRNAs or experiments.

Save test predictions and answers as separate CSV files. **Run all includes this cell.**


In [ ]:
test_data = pd.read_csv(
    data_url + 'AGO2_CLASH_Hejret2023_test.tsv.gz?download=1',
    sep='	', compression='gzip', usecols=input_columns,
)
assert set(test_data['label']) == {0, 1}
assert not test_data.duplicated(['noncodingRNA', 'gene']).any()
assert not set(zip(benchmark_train['noncodingRNA'], benchmark_train['gene'])) & set(
    zip(test_data['noncodingRNA'], test_data['gene']))
test_codes = encode_pairs(test_data)
y_test = test_data['label'].to_numpy()
test_loader = DataLoader(
    TensorDataset(test_codes, torch.tensor(y_test, dtype=torch.float32)),
    batch_size=batch_size, shuffle=False,
)
test_probabilities = predict_probabilities(cnn, test_loader)
test_results = calculate_metrics(y_test, test_probabilities)
print('Official test examples:', len(test_data))
print(test_results.round(4).to_string())
plot_evaluation(y_test, test_probabilities, 'Official test: 2D CNN')
test_ids = 'test_' + test_data.index.astype(str)
pd.DataFrame({'sample_id': test_ids, 'probability': test_probabilities}).to_csv(
    output_dir / 'test_predictions.csv', index=False)
pd.DataFrame({'sample_id': test_ids, 'label': y_test}).to_csv(
    output_dir / 'test_answers.csv', index=False)
print('Prediction and answer files saved in:', output_dir)
